# FinPortfolio — Análise Exploratória e Otimização de Portfólios (Markowitz MPT)

Este notebook demonstra os fundamentos matemáticos e a aplicação prática da **Teoria Moderna de Portfólios de Harry Markowitz** para ativos do mercado brasileiro (B3).

### Etapas:
1. Coleta e Limpeza de Dados de Preços com `yfinance`
2. Cálculo de Retornos Diários, Acumulados e Anualizados
3. Cálculo de Matrizes de Covariância $\mathbf{\Sigma}$ e Correlação $\mathbf{R}$
4. Otimização Quadrática via SciPy (Mínima Volatilidade e Máximo Sharpe)
5. Simulação Monte Carlo e Fronteira Eficiente
6. Backtest Histórico comparando com `BOVA11`

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.express as px

# Adicionar pasta raiz do projeto
sys.path.append(str(Path.cwd().parent))

from src.data import fetch_historical_data, normalize_prices
from src.returns import calculate_daily_returns, calculate_annualized_returns
from src.risk import calculate_volatility, calculate_covariance_matrix, calculate_correlation_matrix
from src.optimization import optimize_gmv, optimize_max_sharpe, calculate_efficient_frontier
from src.portfolio import simulate_random_portfolios
from src.backtest import run_historical_backtest

## 1. Coleta e Limpeza de Dados

In [ ]:
tickers = ['PETR4.SA', 'VALE3.SA', 'ITUB4.SA', 'WEGE3.SA']
benchmark = 'BOVA11.SA'
start_date = '2021-01-01'
end_date = '2024-01-01'

prices = fetch_historical_data(tickers + [benchmark], start_date, end_date)
asset_prices = prices[tickers]
bench_prices = prices[benchmark]

print(f"Linhas baixadas: {len(asset_prices)}")
asset_prices.head()

## 2. Retornos e Riscos Anualizados

In [ ]:
daily_ret = calculate_daily_returns(asset_prices)
ann_ret = calculate_annualized_returns(daily_ret)
daily_vol, ann_vol = calculate_volatility(daily_ret)
cov_mat = calculate_covariance_matrix(daily_ret)
corr_mat = calculate_correlation_matrix(daily_ret)

summary = pd.DataFrame({
    'Retorno Anualizado (%)': ann_ret * 100,
    'Volatilidade Anualizada (%)': ann_vol * 100
})
summary

## 3. Otimização de Markowitz

In [ ]:
rf = 0.1075  # Taxa Selic 10.75%
gmv = optimize_gmv(ann_ret, cov_mat)
max_sharpe = optimize_max_sharpe(ann_ret, cov_mat, risk_free_rate=rf)

print("=== Carteira Mínima Volatilidade (GMV) ===")
print(f"Retorno: {gmv['return']*100:.2f}%")
print(f"Volatilidade: {gmv['volatility']*100:.2f}%")
print(f"Pesos:\n{gmv['weights']}\n")

print("=== Carteira Máximo Sharpe ===")
print(f"Retorno: {max_sharpe['return']*100:.2f}%")
print(f"Volatilidade: {max_sharpe['volatility']*100:.2f}%")
print(f"Sharpe Ratio: {max_sharpe['sharpe']:.2f}")
print(f"Pesos:\n{max_sharpe['weights']}")

## 4. Simulação Monte Carlo e Fronteira Eficiente

In [ ]:
simulations_df, _ = simulate_random_portfolios(ann_ret, cov_mat, num_portfolios=5000, risk_free_rate=rf)
frontier_df = calculate_efficient_frontier(ann_ret, cov_mat, num_points=50, risk_free_rate=rf)

print(f"Portfólios simulados: {len(simulations_df)}")
print(f"Pontos na Fronteira Eficiente: {len(frontier_df)}")

## 5. Backtest Histórico

In [ ]:
bench_ret = calculate_daily_returns(bench_prices.to_frame())['BOVA11.SA']
backtest_res = run_historical_backtest(
    daily_returns=daily_ret,
    optimized_weights=max_sharpe['weights'],
    benchmark_returns=bench_ret,
    risk_free_rate=rf,
    initial_investment=10000.0
)

backtest_res['metrics']